<a href="https://colab.research.google.com/github/Guynos/portfolio/blob/main/STATS_EFFECTIFS_CCC_V1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

# Import des bibliothèques

import pandas as pd
import numpy as np
import openpyxl
import re
from google.colab import files
from datetime import datetime

print("Environnement prêt.")


Environnement prêt.


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Préparation des données

In [3]:

# Importation de l'export des effectifs
print("Importer l'export des effectifs.")

fichiers = files.upload()

fichier_effectifs = None
fichier_calendrier = "/CALENDRIER_BDD.xlsx"

for nom in fichiers.keys():
    if nom.lower().endswith((".xlsx", ".xls")):
        fichier_effectifs = nom
        break

if fichier_effectifs is None:
    raise ValueError("Export des effectifs introuvable.")

import os

# Vérification de la présence du calendrier
if not os.path.exists(fichier_calendrier):
    raise FileNotFoundError(
        f"Calendrier introuvable : {fichier_calendrier}"
    )

# Chargement des deux fichiers Excel
df_brut = pd.read_excel(
    fichier_effectifs,
    header=None
)

df_calendrier = pd.read_excel(
    fichier_calendrier
)

print("\n===== FICHIERS CHARGÉS =====")
print("Export :", fichier_effectifs)
print("Calendrier :", fichier_calendrier)

print("\nLignes de l'export :", len(df_brut))
print("Lignes du calendrier :", len(df_calendrier))
print("Colonnes du calendrier :", list(df_calendrier.columns))


Importer l'export des effectifs.


Saving e15_nov.xlsx to e15_nov (5).xlsx

===== FICHIERS CHARGÉS =====
Export : e15_nov (5).xlsx
Calendrier : /CALENDRIER_BDD.xlsx

Lignes de l'export : 3749
Lignes du calendrier : 1003
Colonnes du calendrier : ['Date', 'YEAR', 'MONTH', 'NUM_DAY', 'DAYS', 'NUM_BY_DAY', 'HOLIDAYS', 'FERIE', 'COMENTS']


In [4]:
# Lecture du DataFrame (df) avec affichage des 35 premières lignes + df shape
df_brut = pd.read_excel(
    fichier_effectifs,
    sheet_name=0,
    header=None
)

print("Dimensions :", df_brut.shape)

display(df_brut.head(35))

Dimensions : (3749, 32)


,0,1,2,3,4,5,6,7,8,9,...,22,23,24,25,26,27,28,29,30,31
0,EFFECTIFS REEL,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Du 01/11/2026 Au 30/11/2026,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,CONTRAT : BUSSY SAINT GEORGES,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,PRESTATION : DEJEUNER,1.0,2.0,3.0,4.0,5.0,6.0,7.0,8.0,9.0,...,22.0,23.0,24.0,25.0,26.0,27.0,28.0,29.0,30.0,TOTAL
7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,Restaurant : ANTOINE GIROUST,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,Adulte,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,1.0,1.0,NaN,1.0,1.0,NaN,NaN,1.0,10


# Controle, vérification & nettoyage

In [5]:
# Vérifacation et controle des prestas - convives (variantes orthograhiques)
# Inventaire des prestations
colonne_libelles = (
    df_brut.iloc[:, 0]
    .dropna()
    .astype(str)
    .str.strip()
)

prestations = sorted({
    valeur.split(":", 1)[1].strip()
    for valeur in colonne_libelles
    if valeur.upper().startswith("PRESTATION :")
})

# Inventaire des lignes de profils et autres libellés
profils = sorted({
    valeur
    for valeur in colonne_libelles
    if not valeur.upper().startswith((
        "CONTRAT",
        "PRESTATION",
        "RESTAURANT",
        "TOTAL",
        "TOUS",
        "TOUTES",
        "DU "
    ))
    and valeur != "EFFECTIFS REEL"
})

print("===== PRESTATIONS DÉTECTÉES =====")
for prestation in prestations:
    print("-", prestation)

print("\n===== LIBELLÉS DÉTECTÉS =====")
for profil in profils:
    print("-", profil)

print("\nNombre de prestations :", len(prestations))
print("Nombre de libellés :", len(profils))


===== PRESTATIONS DÉTECTÉES =====
- COLLATION
- DEJEUNER
- DEJEUNER B
- DINER
- GOUTER
- GOUTER 2
- INGREDIENTS NON FACTURES
- PETIT DEJEUNER
- PIQUE-NIQUE 1
- PRESTATION ANNEXE FACTUREE 1
- REPAS DE SECOURS

===== LIBELLÉS DÉTECTÉS =====
- 1 Adulte individuel
- 6 Adulte
- Ad indiv 1
- Ad indiv s/viande.
- Adulte
- Adulte Elior
- Adulte Elior S/viande
- Adulte S/Viande
- Adulte communal
- Adulte ind
- Adulte indiv Sans Viande
- Adulte individuel
- Adulte individuel Elior
- Adulte individuel S/Viande
- Adultes
- Adultes Elior
- Adultes Elior S/Viande
- Adultes Enseignant
- Adultes S/Viande
- Elémentaire
- Elémentaires
- Elémentaires S/Viande
- Enseignant
- Maternelle
- Maternelle S/Viande
- Maternelles
- Maternelles S/Viande
- Plateau témoin
- Portage
- Portage S/Sucre
- Portage léger en sel
- Portage mixé
- Reajustement
- Réajustement
- Réajustement végétarien
- Sans Porc

Nombre de prestations : 11
Nombre de libellés : 36


In [6]:
# Nettoyage, normalisation des convives - detextion des accents

import unicodedata

def normaliser_libelle(valeur):
    texte = str(valeur).strip().lower()
    texte = unicodedata.normalize("NFKD", texte)
    texte = "".join(
        c for c in texte
        if not unicodedata.combining(c)
    )
    return texte

correspondance_profils = {
    "maternelle": "Maternelles avec viande",
    "maternelles": "Maternelles avec viande",
    "maternelle s/viande": "Maternelles sans viande",
    "maternelles s/viande": "Maternelles sans viande",

    "elementaire": "Élémentaires avec viande",
    "elementaires": "Élémentaires avec viande",
    "elementaire s/viande": "Élémentaires sans viande",
    "elementaires s/viande": "Élémentaires sans viande",
}

lignes_profils = []

for index, valeur in df_brut.iloc[:, 0].items():

    if pd.isna(valeur):
        continue

    libelle = normaliser_libelle(valeur)

    if libelle in correspondance_profils:
        lignes_profils.append({
            "Ligne Excel": index + 1,
            "Libellé original": str(valeur).strip(),
            "Profil identifié": correspondance_profils[libelle]
        })

df_profils = pd.DataFrame(lignes_profils)

print("Nombre de lignes reconnues :", len(df_profils))

display(
    df_profils.groupby(
        ["Profil identifié", "Libellé original"]
    ).size().reset_index(name="Nombre de lignes")
)


Nombre de lignes reconnues : 753


,Profil identifié,Libellé original,Nombre de lignes
0,Maternelles avec viande,Maternelle,121
1,Maternelles avec viande,Maternelles,157
2,Maternelles sans viande,Maternelle S/Viande,5
3,Maternelles sans viande,Maternelles S/Viande,78
4,Élémentaires avec viande,Elémentaire,139
5,Élémentaires avec viande,Elémentaires,175
6,Élémentaires sans viande,Elémentaires S/Viande,78


In [7]:

# ============================================================
# EXTRACTION DES EFFECTIFS DEJEUNER
# ============================================================


from datetime import date

# Détection de la période du fichier
texte_periode = str(df_brut.iloc[2, 0])

match_periode = re.search(
    r"Du\s+(\d{2})/(\d{2})/(\d{4})\s+Au",
    texte_periode,
    re.IGNORECASE
)

if not match_periode:
    raise ValueError("Impossible de détecter la période du fichier.")

jour_debut, mois, annee = map(int, match_periode.groups())


# Extraction des données
donnees = []

ville = None
prestation = None
ecole = None

for index, ligne in df_brut.iterrows():

    valeur = ligne.iloc[0]

    if pd.isna(valeur):
        continue

    libelle = str(valeur).strip()
    libelle_normalise = normaliser_libelle(libelle)

    # Détection de la ville
    if libelle_normalise.startswith("contrat :"):
        ville = libelle.split(":", 1)[1].strip()
        prestation = None
        ecole = None
        continue

    # Détection de la prestation
    if libelle_normalise.startswith("prestation :"):
        prestation = libelle.split(":", 1)[1].strip()
        ecole = None
        continue

    # Détection de l'école
    if libelle_normalise.startswith("restaurant :"):
        ecole = libelle.split(":", 1)[1].strip()
        continue

    # Détection des lignes de total : elles ne sont jamais des profils
    if libelle_normalise.startswith("total restaurant :"):
        ecole = None
        continue

    if libelle_normalise.startswith("total prestation :"):
        ecole = None
        continue

    # On ne conserve que les prestations DEJEUNER
    if prestation is None or not normaliser_libelle(prestation).startswith("dejeuner"):
        continue

    if ville is None or ecole is None:
        continue

    # Identification du profil
    profil = correspondance_profils.get(libelle_normalise)

    if profil is None:
        continue

    # Recherche de la ligne contenant les numéros des jours
    ligne_entete = df_brut.iloc[
        df_brut.index[
            (df_brut.iloc[:, 0].astype(str)
             == f"PRESTATION : {prestation}")
        ][0]
    ]

    # Lecture des colonnes B à AE uniquement
    # La colonne AF contient le total et ne doit pas être comptabilisée
    for colonne in range(1, 31):

        jour = ligne_entete.iloc[colonne]

        if pd.isna(jour):
            continue

        jour = int(jour)
        date_jour = date(annee, mois, jour)

        effectif = pd.to_numeric(
            ligne.iloc[colonne],
            errors="coerce"
        )

        effectif = 0 if pd.isna(effectif) else float(effectif)

        donnees.append({
            "Ville": ville,
            "Ecole": ecole,
            "Date": date_jour,
            "Prestation": prestation,
            "Profil": profil,
            "Effectif": effectif
        })


# Création du DataFrame d'extraction
df_extraction = pd.DataFrame(donnees)

print("Extraction terminée.")
print(f"Nombre de lignes extraites : {len(df_extraction)}")
print(f"Nombre de villes : {df_extraction['Ville'].nunique()}")
print(f"Nombre d'écoles : {df_extraction['Ecole'].nunique()}")


Extraction terminée.
Nombre de lignes extraites : 10350
Nombre de villes : 15
Nombre d'écoles : 156


In [8]:

print("===== RÉPARTITION PAR PROFIL =====")

display(
    df_extraction.groupby("Profil")
    .agg(
        Nombre_de_lignes=("Effectif", "count"),
        Total_effectifs=("Effectif", "sum")
    )
    .reset_index()
)

print("\n===== RÉPARTITION PAR VILLE =====")

display(
    df_extraction.groupby("Ville")
    .agg(
        Nombre_ecoles=("Ecole", "nunique"),
        Nombre_lignes=("Effectif", "count")
    )
    .reset_index()
)

print("\n===== CONTRÔLE DES PRESTATIONS =====")

display(
    df_extraction.groupby("Prestation")
    .agg(
        Nombre_lignes=("Effectif", "count"),
        Total_effectifs=("Effectif", "sum")
    )
    .reset_index()
)


===== RÉPARTITION PAR PROFIL =====


,Profil,Nombre_de_lignes,Total_effectifs
0,Maternelles avec viande,3450,113481.0
1,Maternelles sans viande,1800,11770.0
2,Élémentaires avec viande,3300,189363.0
3,Élémentaires sans viande,1800,20278.0



===== RÉPARTITION PAR VILLE =====


,Ville,Nombre_ecoles,Nombre_lignes
0,BUSSY SAINT GEORGES,15,1800
1,CHELLES,23,1110
2,COUILLY PONT AUX DAMES,2,120
3,DAMPMART,2,240
4,EMERAINVILLE,7,360
5,ENNERY SCO,2,60
6,FONTENAY PARISIS,3,240
7,LAGNY-SUR-MARNES,13,1200
8,LE RAINCY.,12,720
9,LOGNES.,7,420



===== CONTRÔLE DES PRESTATIONS =====


,Prestation,Nombre_lignes,Total_effectifs
0,DEJEUNER,10350,334892.0


In [9]:

# ============================================================
# FILTRAGE DES JOURS SCOLAIRES
# ============================================================

# Copie du calendrier
df_cal = df_calendrier.copy()

# Normalisation des dates
df_cal["Date"] = pd.to_datetime(df_cal["Date"], errors="coerce")
df_extraction["Date"] = pd.to_datetime(df_extraction["Date"])

# Vérification des dates manquantes dans le calendrier
dates_export = set(df_extraction["Date"].unique())
dates_calendrier = set(df_cal["Date"].dropna().unique())

dates_manquantes = dates_export - dates_calendrier

if dates_manquantes:
    raise ValueError(
        f"Dates absentes du calendrier : {sorted(dates_manquantes)}"
    )

# Vérification des doublons
if df_cal["Date"].duplicated().any():
    raise ValueError("Le calendrier contient des dates en doublon.")

# Normalisation des indicateurs
df_cal["HOLIDAYS_NORM"] = df_cal["HOLIDAYS"].apply(normaliser_libelle)
df_cal["FERIE_NORM"] = df_cal["FERIE"].apply(normaliser_libelle)

# Jours autorisés
jours_autorises = [
    "lundi",
    "mardi",
    "jeudi",
    "vendredi"
]

df_cal["JOUR_NORM"] = df_cal["DAYS"].apply(normaliser_libelle)

# Conditions de conservation
df_cal["JOUR_VALIDE"] = (
    df_cal["JOUR_NORM"].isin(jours_autorises)
    & (df_cal["HOLIDAYS_NORM"] == "non")
    & (df_cal["FERIE_NORM"] == "non")
)

# Association du calendrier aux effectifs
df_analyse = df_extraction.merge(
    df_cal[["Date", "DAYS", "HOLIDAYS", "FERIE", "JOUR_VALIDE"]],
    on="Date",
    how="left",
    validate="many_to_one"
)

# Conservation des seules journées scolaires autorisées
df_analyse = df_analyse[
    df_analyse["JOUR_VALIDE"]
].copy()

print("===== BILAN DU FILTRAGE =====")
print("Dates dans l'export :", len(dates_export))
print("Jours conservés :", df_analyse["Date"].nunique())
print("Lignes conservées :", len(df_analyse))

print("\n===== RÉPARTITION PAR JOUR =====")

display(
    df_analyse.groupby("DAYS")["Date"]
    .nunique()
    .reindex(["Lundi", "Mardi", "Jeudi", "Vendredi"])
    .fillna(0)
    .astype(int)
    .reset_index(name="Nombre de dates")
)

print("\n===== PREMIÈRES DONNÉES CONSERVÉES =====")
display(df_analyse.head(15))


===== BILAN DU FILTRAGE =====
Dates dans l'export : 30
Jours conservés : 16
Lignes conservées : 5520

===== RÉPARTITION PAR JOUR =====


,DAYS,Nombre de dates
0,Lundi,4
1,Mardi,4
2,Jeudi,4
3,Vendredi,4



===== PREMIÈRES DONNÉES CONSERVÉES =====


,Ville,Ecole,Date,Prestation,Profil,Effectif,DAYS,HOLIDAYS,FERIE,JOUR_VALIDE
2,BUSSY SAINT GEORGES,ANTOINE GIROUST,2026-11-03,DEJEUNER,Élémentaires avec viande,117.0,Mardi,Non,Non,True
4,BUSSY SAINT GEORGES,ANTOINE GIROUST,2026-11-05,DEJEUNER,Élémentaires avec viande,118.0,Jeudi,Non,Non,True
5,BUSSY SAINT GEORGES,ANTOINE GIROUST,2026-11-06,DEJEUNER,Élémentaires avec viande,116.0,Vendredi,Non,Non,True
8,BUSSY SAINT GEORGES,ANTOINE GIROUST,2026-11-09,DEJEUNER,Élémentaires avec viande,116.0,Lundi,Non,Non,True
9,BUSSY SAINT GEORGES,ANTOINE GIROUST,2026-11-10,DEJEUNER,Élémentaires avec viande,119.0,Mardi,Non,Non,True
11,BUSSY SAINT GEORGES,ANTOINE GIROUST,2026-11-12,DEJEUNER,Élémentaires avec viande,118.0,Jeudi,Non,Non,True
12,BUSSY SAINT GEORGES,ANTOINE GIROUST,2026-11-13,DEJEUNER,Élémentaires avec viande,104.0,Vendredi,Non,Non,True
15,BUSSY SAINT GEORGES,ANTOINE GIROUST,2026-11-16,DEJEUNER,Élémentaires avec viande,104.0,Lundi,Non,Non,True
16,BUSSY SAINT GEORGES,ANTOINE GIROUST,2026-11-17,DEJEUNER,Élémentaires avec viande,104.0,Mardi,Non,Non,True
18,BUSSY SAINT GEORGES,ANTOINE GIROUST,2026-11-19,DEJEUNER,Élémentaires avec viande,104.0,Jeudi,Non,Non,True


In [10]:
# Calcul des statistiques par ville, école, jour et profil

statistiques = (
    df_analyse
    .groupby(
        ["Ville", "Ecole", "DAYS", "Profil"],
        as_index=False
    )
    .agg(
        Nombre_jours=("Effectif", "count"),
        Minimum=("Effectif", "min"),
        Moyenne=("Effectif", "mean"),
        Maximum=("Effectif", "max"),
        Ecart_type=("Effectif", lambda x: x.std(ddof=0))
    )
)

# Arrondir les résultats à 2 décimales
colonnes_stats = ["Minimum", "Moyenne", "Maximum", "Ecart_type"]
statistiques[colonnes_stats] = statistiques[colonnes_stats].round(2)

print("Nombre de lignes statistiques :", len(statistiques))
print("Nombre de villes :", statistiques["Ville"].nunique())
print("Nombre d'écoles :", statistiques[["Ville", "Ecole"]].drop_duplicates().shape[0])

display(statistiques.head(20))

Nombre de lignes statistiques : 1380
Nombre de villes : 15
Nombre d'écoles : 157


,Ville,Ecole,DAYS,Profil,Nombre_jours,Minimum,Moyenne,Maximum,Ecart_type
0,BUSSY SAINT GEORGES,ANTOINE GIROUST,Jeudi,Maternelles avec viande,4,50.0,53.75,59.0,3.90
1,BUSSY SAINT GEORGES,ANTOINE GIROUST,Jeudi,Maternelles sans viande,4,0.0,3.00,6.0,3.00
2,BUSSY SAINT GEORGES,ANTOINE GIROUST,Jeudi,Élémentaires avec viande,4,104.0,111.00,118.0,7.00
3,BUSSY SAINT GEORGES,ANTOINE GIROUST,Jeudi,Élémentaires sans viande,4,0.0,7.00,14.0,7.00
4,BUSSY SAINT GEORGES,ANTOINE GIROUST,Lundi,Maternelles avec viande,4,50.0,51.50,56.0,2.60
5,BUSSY SAINT GEORGES,ANTOINE GIROUST,Lundi,Maternelles sans viande,4,0.0,4.50,6.0,2.60
6,BUSSY SAINT GEORGES,ANTOINE GIROUST,Lundi,Élémentaires avec viande,4,104.0,107.00,116.0,5.20
7,BUSSY SAINT GEORGES,ANTOINE GIROUST,Lundi,Élémentaires sans viande,4,0.0,10.50,14.0,6.06
8,BUSSY SAINT GEORGES,ANTOINE GIROUST,Mardi,Maternelles avec viande,4,50.0,52.75,56.0,2.77
9,BUSSY SAINT GEORGES,ANTOINE GIROUST,Mardi,Maternelles sans viande,4,0.0,3.00,6.0,3.00


# Construction du dashboard

In [11]:
# ==========================================
# ÉTAPE 1 - PRÉPARATION DU TABLEAU DE BORD
# ==========================================

import ipywidgets as widgets
import plotly.express as px
from IPython.display import display, clear_output

# Vérification des données nécessaires
if "df_analyse" not in globals():
    raise NameError(
        "df_analyse est introuvable. "
        "Exécute d'abord les cellules d'extraction et de filtrage."
    )

# Calcul des statistiques si nécessaire
if "statistiques" not in globals():

    statistiques = (
        df_analyse
        .groupby(
            ["Ville", "Ecole", "DAYS", "Profil"],
            as_index=False
        )
        .agg(
            Nombre_jours=("Effectif", "count"),
            Minimum=("Effectif", "min"),
            Moyenne=("Effectif", "mean"),
            Maximum=("Effectif", "max"),
            Ecart_type=("Effectif", lambda x: x.std(ddof=0))
        )
    )

    statistiques[
        ["Minimum", "Moyenne", "Maximum", "Ecart_type"]
    ] = statistiques[
        ["Minimum", "Moyenne", "Maximum", "Ecart_type"]
    ].round(2)

# Vérification des colonnes
colonnes_attendues = [
    "Ville", "Ecole", "DAYS", "Profil",
    "Nombre_jours", "Minimum", "Moyenne",
    "Maximum", "Ecart_type"
]

colonnes_manquantes = [
    c for c in colonnes_attendues
    if c not in statistiques.columns
]

if colonnes_manquantes:
    raise ValueError(
        f"Colonnes manquantes : {colonnes_manquantes}"
    )

print("Préparation réussie.")
print("Nombre de villes :", statistiques["Ville"].nunique())
print("Nombre d'écoles :", statistiques[["Ville", "Ecole"]].drop_duplicates().shape[0])
print("Nombre de lignes statistiques :", len(statistiques))
print("Bibliothèques graphiques chargées.")

Préparation réussie.
Nombre de villes : 15
Nombre d'écoles : 157
Nombre de lignes statistiques : 1380
Bibliothèques graphiques chargées.


In [12]:
!pip install -U plotly

In [13]:
# ==========================================
# ÉTAPE 2 - FILTRES INTERACTIFS
# ==========================================

villes = ["Toutes les villes"] + sorted(
    statistiques["Ville"].dropna().unique().tolist()
)

filtre_ville = widgets.Dropdown(
    options=villes,
    value="Toutes les villes",
    description="Ville :",
    style={"description_width": "80px"},
    layout=widgets.Layout(width="400px")
)

filtre_ecole = widgets.Dropdown(
    options=["Toutes les écoles"],
    value="Toutes les écoles",
    description="École :",
    style={"description_width": "80px"},
    layout=widgets.Layout(width="400px")
)

profils = [
    "Tous les profils",
    "Maternelles avec viande",
    "Maternelles sans viande",
    "Élémentaires avec viande",
    "Élémentaires sans viande"
]

filtre_profil = widgets.Dropdown(
    options=profils,
    value="Tous les profils",
    description="Profil :",
    style={"description_width": "80px"},
    layout=widgets.Layout(width="400px")
)

def actualiser_ecoles(change=None):

    if filtre_ville.value == "Toutes les villes":
        ecoles = sorted(
            statistiques["Ecole"].dropna().unique().tolist()
        )
    else:
        ecoles = sorted(
            statistiques.loc[
                statistiques["Ville"] == filtre_ville.value,
                "Ecole"
            ].dropna().unique().tolist()
        )

    anciennes_options = list(filtre_ecole.options)

    nouvelles_options = ["Toutes les écoles"] + ecoles

    filtre_ecole.options = nouvelles_options

    if filtre_ecole.value not in nouvelles_options:
        filtre_ecole.value = "Toutes les écoles"

filtre_ville.observe(actualiser_ecoles, names="value")

actualiser_ecoles()

# Affichage des filtres
display(
    widgets.VBox([
        widgets.HTML("<h3>Filtres du tableau de bord</h3>"),
        filtre_ville,
        filtre_ecole,
        filtre_profil
    ])
)

In [15]:
# ==========================================
# ÉTAPE 3 - INDICATEURS ET GRAPHIQUES (version corrigée)
# ==========================================

import plotly.express as px
from IPython.display import display, clear_output
import ipywidgets as widgets

jours_ordre = ["Lundi", "Mardi", "Jeudi", "Vendredi"]

# Conteneur dédié pour le dashboard (évite les problèmes de clear_output)
output_dashboard = widgets.Output()

def afficher_dashboard(change=None):
    with output_dashboard:
        clear_output(wait=True)

        # APPLICATION DES FILTRES
        df = df_analyse.copy()

        if filtre_ville.value != "Toutes les villes":
            df = df[df["Ville"] == filtre_ville.value]

        if filtre_ecole.value != "Toutes les écoles":
            df = df[df["Ecole"] == filtre_ecole.value]

        if filtre_profil.value != "Tous les profils":
            df = df[df["Profil"] == filtre_profil.value]

        if df.empty:
            print("Aucune donnée disponible pour cette sélection.")
            return

        # ---- INDICATEURS ----
        moyenne = df["Effectif"].mean()
        minimum = df["Effectif"].min()
        maximum = df["Effectif"].max()
        ecart_type = df["Effectif"].std(ddof=0)

        indicateurs = widgets.HBox([
            widgets.HTML(f"<div style='padding:12px;background:#e8f1fa;border-radius:8px;width:150px'><small>Effectif moyen</small><h2>{moyenne:.1f}</h2></div>"),
            widgets.HTML(f"<div style='padding:12px;background:#e8f1fa;border-radius:8px;width:150px'><small>Minimum</small><h2>{minimum:.0f}</h2></div>"),
            widgets.HTML(f"<div style='padding:12px;background:#e8f1fa;border-radius:8px;width:150px'><small>Maximum</small><h2>{maximum:.0f}</h2></div>"),
            widgets.HTML(f"<div style='padding:12px;background:#e8f1fa;border-radius:8px;width:150px'><small>Écart-type</small><h2>{ecart_type:.1f}</h2></div>")
        ])
        display(indicateurs)

        # ---- GRAPHIQUE 1 : par jour ----
        graphique_jours = (
            df.groupby("DAYS", as_index=False)
            .agg(Effectif_moyen=("Effectif", "mean"))
        )
        graphique_jours["DAYS"] = pd.Categorical(graphique_jours["DAYS"], categories=jours_ordre, ordered=True)
        graphique_jours = graphique_jours.sort_values("DAYS")

        fig_jours = px.bar(
            graphique_jours, x="DAYS", y="Effectif_moyen",
            title="Effectif moyen par jour de la semaine",
            labels={"DAYS": "Jour", "Effectif_moyen": "Effectif moyen"},
            text_auto=".1f"
        )
        fig_jours.update_layout(template="plotly_white", height=400)
        fig_jours.show()   # ← mieux que display() pour Plotly

        # ---- GRAPHIQUE 2 : par profil ----
        graphique_profils = (
            df.groupby("Profil", as_index=False)
            .agg(Effectif_moyen=("Effectif", "mean"))
        )
        fig_profils = px.bar(
            graphique_profils, x="Profil", y="Effectif_moyen",
            title="Comparaison des effectifs moyens par profil",
            labels={"Profil": "Profil", "Effectif_moyen": "Effectif moyen"},
            text_auto=".1f"
        )
        fig_profils.update_layout(template="plotly_white", height=450, xaxis_tickangle=-20)
        fig_profils.show()

        # ---- GRAPHIQUE 3 : par école (limité pour éviter les hauteurs énormes) ----
        effectifs_ecoles = (
            df.groupby(["Ville", "Ecole", "Date"], as_index=False)
            .agg(Effectif_total=("Effectif", "sum"))
        )
        graphique_ecoles = (
            effectifs_ecoles
            .groupby(["Ville", "Ecole"], as_index=False)
            .agg(Effectif_moyen=("Effectif_total", "mean"))
        )
        graphique_ecoles["Ecole_label"] = graphique_ecoles["Ville"] + " - " + graphique_ecoles["Ecole"]
        graphique_ecoles = graphique_ecoles.sort_values("Effectif_moyen", ascending=True)

        # Limitation de hauteur + éventuelle restriction si trop d'écoles
        n_ecoles = len(graphique_ecoles)
        if n_ecoles > 40:
            # On n'affiche que les 40 plus grosses pour rester lisible
            graphique_ecoles = graphique_ecoles.tail(40)
            titre = f"Top 40 écoles (sur {n_ecoles}) – Effectif moyen journalier"
        else:
            titre = "Comparaison des effectifs moyens par école"

        fig_ecoles = px.bar(
            graphique_ecoles,
            x="Effectif_moyen", y="Ecole_label",
            orientation="h",
            title=titre,
            labels={"Ecole_label": "École", "Effectif_moyen": "Effectif moyen journalier"},
            text_auto=".1f"
        )
        fig_ecoles.update_layout(
            template="plotly_white",
            height=max(450, len(graphique_ecoles) * 28),
            yaxis_title="",
            xaxis_title="Effectif moyen journalier",
            margin=dict(l=250)  # plus d'espace pour les noms d'écoles
        )
        fig_ecoles.show()

# On n'attache les observers qu'une seule fois
try:
    filtre_ville.unobserve(afficher_dashboard, names="value")
    filtre_ecole.unobserve(afficher_dashboard, names="value")
    filtre_profil.unobserve(afficher_dashboard, names="value")
except:
    pass

filtre_ville.observe(afficher_dashboard, names="value")
filtre_ecole.observe(afficher_dashboard, names="value")
filtre_profil.observe(afficher_dashboard, names="value")

# Affichage du conteneur + premier rendu
display(output_dashboard)
afficher_dashboard()

Output()